# PTB-XL ECG EDA

Notebook đầu tiên để khám phá dữ liệu ECG PTB-XL trước khi xây dựng mô hình phân loại đa nhãn.

Mục tiêu:
- kiểm tra schema dữ liệu metadata
- quan sát phân bố tuổi, giới tính và nhãn chẩn đoán
- hiểu cách mã hóa nhãn `scp_codes`
- chuẩn bị tiền xử lý cho bước mô hình sau này

In [ ]:
from pathlib import Path
import ast
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('default')

current_dir = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in (current_dir, *current_dir.parents)
     if (candidate / 'data' / 'ptbxl_database.csv').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate data/ptbxl_database.csv from this notebook working directory.')
data_dir = project_root / 'data'
meta_path = data_dir / 'ptbxl_database.csv'
meta = pd.read_csv(meta_path, encoding='latin-1')
meta.head()

In [ ]:
print(f"Số dòng: {meta.shape[0]}")
print(f"Số cột: {meta.shape[1]}")
print()
print(meta.dtypes.to_string())
print()
print('Các cột quan trọng:', [c for c in meta.columns[:20]])

In [ ]:
# Lọc các cột số và kiểm tra missing values
numeric_cols = meta.select_dtypes(include=['number']).columns
missing_summary = meta.isna().sum().sort_values(ascending=False)
missing_summary.head(20)

In [ ]:
# Descriptive statistics use the full metadata before quarantine; they are not preprocessing fit values.
n_records = len(meta)
height_cm = pd.to_numeric(meta['height'], errors='coerce')
weight_kg = pd.to_numeric(meta['weight'], errors='coerce')
valid_bmi = height_cm.gt(0) & weight_kg.gt(0)
bmi = pd.Series(float('nan'), index=meta.index, dtype='float64')
bmi.loc[valid_bmi] = weight_kg.loc[valid_bmi] / (height_cm.loc[valid_bmi] / 100) ** 2

continuous_variables = {
    'Age (years)': pd.to_numeric(meta['age'], errors='coerce'),
    'Height (cm)': height_cm,
    'Weight (kg)': weight_kg,
    'BMI (kg/m^2)': bmi,
}
summary_rows = []
for variable, values in continuous_variables.items():
    available = values.dropna()
    q1, median, q3 = available.quantile([0.25, 0.50, 0.75]) if not available.empty else (float('nan'),) * 3
    summary_rows.append({
        'Variable': variable,
        'Available n (%)': f'{len(available)} ({len(available) / n_records * 100:.1f}%)',
        'Missing n (%)': f'{n_records - len(available)} ({(n_records - len(available)) / n_records * 100:.1f}%)',
        'Mean +/- SD': f'{available.mean():.2f} +/- {available.std():.2f}' if not available.empty else 'NA',
        'Median (Q1-Q3)': f'{median:.2f} ({q1:.2f}-{q3:.2f})' if not available.empty else 'NA',
    })

descriptive_summary = pd.DataFrame(summary_rows)
print('DESCRIPTIVE METADATA SUMMARY (full dataset; before quarantine)')
print(descriptive_summary.to_string(index=False))

# PTB-XL sex coding: 0=male, 1=female; missing or other codes are unknown.
sex_codes = meta['sex']
sex_categories = {
    'Male': sex_codes.eq(0),
    'Female': sex_codes.eq(1),
    'Unknown': ~sex_codes.isin([0, 1]),
}
sex_summary = pd.DataFrame([
    {
        'Category': category,
        'n (%)': f'{int(mask.sum())} ({mask.sum() / n_records * 100:.1f}%)',
    }
    for category, mask in sex_categories.items()
])
print('SEX DISTRIBUTION (full dataset)')
print(sex_summary.to_string(index=False))
print(f'BMI available n: {int(valid_bmi.sum())}; formula: weight_kg / (height_cm / 100)^2; no outlier trimming.')


In [ ]:
# Phân bố giới tính và độ tuổi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

meta['sex'].value_counts(dropna=False).plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Phân bố giới tính')
axes[0].set_xlabel('Giới tính (0: male, 1: female)')
axes[0].set_ylabel('Số lượng')

meta['age'].dropna().hist(bins=30, ax=axes[1], edgecolor='black', color='#55A868')
axes[1].set_title('Phân bố độ tuổi')
axes[1].set_xlabel('Tuổi')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.show()

In [ ]:
# Parse chuỗi scp_codes và ánh xạ mã chẩn đoán sang 5 diagnostic superclass

def parse_scp_codes(code_str):
    if pd.isna(code_str):
        return {}
    try:
        parsed = ast.literal_eval(code_str)
        if isinstance(parsed, dict):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return {}

meta['scp_codes_dict'] = meta['scp_codes'].map(parse_scp_codes)

diagnostic_superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
scp_statements = pd.read_csv(data_dir / 'scp_statements.csv', index_col=0)
diagnostic_map = scp_statements.loc[
    scp_statements['diagnostic'].eq(1.0)
    & scp_statements['diagnostic_class'].isin(diagnostic_superclasses),
    'diagnostic_class'
].to_dict()

meta['diagnostic_labels'] = meta['scp_codes_dict'].map(
    lambda codes: sorted(
        {diagnostic_map[code] for code in codes if code in diagnostic_map},
        key=diagnostic_superclasses.index
    )
)
label_counts = pd.Series(
    [label for labels in meta['diagnostic_labels'] for label in labels]
).value_counts().reindex(diagnostic_superclasses, fill_value=0).astype(int)

print(f'Số nhãn chẩn đoán cấp cao: {len(diagnostic_superclasses)}')
label_counts

In [ ]:
# Tạo ma trận multi-label chỉ gồm 5 diagnostic superclass
label_matrix = pd.DataFrame(
    {
        label: meta['diagnostic_labels'].map(lambda labels: int(label in labels))
        for label in diagnostic_superclasses
    },
    index=meta.index
)
label_matrix.head()

In [ ]:
# Pairwise co-occurrence among the five mapped diagnostic superclasses.
cooccurrence_counts = label_matrix[diagnostic_superclasses].T.dot(
    label_matrix[diagnostic_superclasses]
).astype(int)
label_totals = label_matrix[diagnostic_superclasses].sum(axis=0).reindex(diagnostic_superclasses)
cooccurrence_percent = cooccurrence_counts.div(
    label_totals.replace(0, float('nan')), axis=0
).mul(100)

print('Co-occurrence counts (rows and columns are superclass labels):')
print(cooccurrence_counts.to_string())
print('Co-occurrence percent within each row label:')
print(cooccurrence_percent.round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
heatmaps = [
    (cooccurrence_counts.to_numpy(), 'Count of ECG records', 'Records', False),
    (cooccurrence_percent.to_numpy(), 'Percent within row label', 'Percent', True),
]
for axis, (values, title, colorbar_label, is_percent) in zip(axes, heatmaps):
    image = axis.imshow(values, cmap='Blues')
    axis.set_xticks(range(len(diagnostic_superclasses)), labels=diagnostic_superclasses)
    axis.set_yticks(range(len(diagnostic_superclasses)), labels=diagnostic_superclasses)
    axis.set_xlabel('Co-occurring superclass')
    axis.set_ylabel('Reference superclass')
    axis.set_title(title)
    for row_index in range(len(diagnostic_superclasses)):
        for column_index in range(len(diagnostic_superclasses)):
            value = values[row_index, column_index]
            cell_text = f'{value:.1f}%' if is_percent else f'{int(value)}'
            text_color = 'white' if value > values.max() * 0.55 else '#202020'
            axis.text(column_index, row_index, cell_text, ha='center', va='center', color=text_color)
    fig.colorbar(image, ax=axis, shrink=0.8, label=colorbar_label)
fig.suptitle('??ng xu?t hi?n c?a 5 diagnostic superclass')
fig.tight_layout()
plt.show()


In [ ]:
# Frequency of the five diagnostic superclass labels
top_labels = label_counts.rename_axis('label').reset_index(name='count')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_labels['label'], top_labels['count'], color='#4C72B0')
ax.set_title('Tần suất 5 nhóm chẩn đoán ECG')
ax.set_xlabel('Số bản ghi ECG')
ax.set_ylabel('Nhóm chẩn đán')
fig.tight_layout()
plt.show()


In [ ]:
# Tóm tắt số superclass trong 5 nhãn chính trên mỗi ECG
meta['n_labels'] = label_matrix.sum(axis=1)
meta['n_labels'].describe()

In [ ]:
# Distribution of the number of mapped diagnostic superclasses per ECG
fig, ax = plt.subplots(figsize=(8, 5))
bins = [value - 0.5 for value in range(len(diagnostic_superclasses) + 2)]
ax.hist(meta['n_labels'], bins=bins, color='#4C72B0', edgecolor='white')
ax.set_title('Phân bố số superclass trên mỗi bản ghi ECG')
ax.set_xlabel('Số nhóm chẩn đán chính')
ax.set_ylabel('Số bản ghi ECG')
ax.set_xticks(range(len(diagnostic_superclasses) + 1))
ax.set_xlim(-0.5, len(diagnostic_superclasses) + 0.5)
fig.tight_layout()
plt.show()
